In [2]:
import numpy as np
from dataclasses import dataclass

In [3]:
@dataclass
class BearingParams:
    fs: int = 12000
    duration: float = 1.0
    shaft_freq: float = 30.0
    noise_std: float = 0.03
    harmonic_decay: float = 0.55


class BearingSignalSimulator:
    def __init__(self, params: BearingParams = BearingParams()):
        self.params = params
        self.t = np.arange(0, params.duration, 1 / params.fs)

    def _healthy_base(self, amp=1.0, phase=0.0, speed_scale=1.0):
        f0 = self.params.shaft_freq * speed_scale
        x = np.zeros_like(self.t)
        for k in range(1, 6):
            x += (amp * (self.params.harmonic_decay ** (k - 1))) * np.sin(2 * np.pi * k * f0 * self.t + phase / k)
        return x

    def _impulse_train(self, fault_freq, strength=0.5, jitter=0.0):
        period = 1.0 / fault_freq
        n_pulses = int(self.params.duration / period) + 2
        pulse_times = np.arange(n_pulses) * period
        if jitter > 0:
            pulse_times = pulse_times + np.random.normal(0, jitter * period, size=pulse_times.shape)
        pulse_times = pulse_times[(pulse_times >= 0) & (pulse_times < self.params.duration)]

        sig = np.zeros_like(self.t)
        width = int(0.001 * self.params.fs)
        width = max(width, 3)
        kernel_t = np.linspace(-2, 2, width)
        kernel = np.exp(-kernel_t**2) * np.cos(12 * kernel_t)
        kernel = kernel / (np.max(np.abs(kernel)) + 1e-8)

        for pt in pulse_times:
            idx = int(pt * self.params.fs)
            left = max(0, idx - width // 2)
            right = min(len(sig), left + width)
            k = kernel[: right - left]
            sig[left:right] += strength * k
        return sig

    def _apply_nuisance(self, x, amp_scale=1.0, offset=0.0, extra_noise=0.0, local_dropout=0.0):
        y = amp_scale * x + offset
        if local_dropout > 0:
            mask = np.ones_like(y)
            n = len(y)
            span = int(local_dropout * n)
            if span > 0:
                start = np.random.randint(0, max(1, n - span))
                mask[start:start + span] *= np.random.uniform(0.6, 1.0)
            y = y * mask
        y = y + np.random.normal(0, self.params.noise_std + extra_noise, size=len(y))
        return y

    def sample(self, regime="healthy", mc=True):
        amp = np.random.uniform(0.85, 1.15) if mc else 1.0
        phase = np.random.uniform(-0.3, 0.3) if mc else 0.0
        speed_scale = np.random.uniform(0.95, 1.05) if mc else 1.0
        offset = np.random.uniform(-0.05, 0.05) if mc else 0.0
        extra_noise = np.random.uniform(0.0, 0.03) if mc else 0.0
        dropout = np.random.uniform(0.0, 0.03) if mc else 0.0

        base = self._healthy_base(amp=amp, phase=phase, speed_scale=speed_scale)

        if regime == "healthy":
            x = base
        elif regime == "outer_race":
            x = base + self._impulse_train(fault_freq=108 * speed_scale, strength=np.random.uniform(0.35, 0.7), jitter=0.05)
        elif regime == "inner_race":
            x = base + self._impulse_train(fault_freq=162 * speed_scale, strength=np.random.uniform(0.35, 0.8), jitter=0.03)
        elif regime == "ball_fault":
            mod = 1 + 0.25 * np.sin(2 * np.pi * 12 * self.t)
            x = base + mod * self._impulse_train(fault_freq=72 * speed_scale, strength=np.random.uniform(0.3, 0.6), jitter=0.06)
        else:
            raise ValueError(f"Unknown regime: {regime}")

        return self._apply_nuisance(x, amp_scale=1.0, offset=offset, extra_noise=extra_noise, local_dropout=dropout)

    def make_dataset(self, n_per_class=200):
        regimes = ["healthy", "outer_race", "inner_race", "ball_fault"]
        X, y = [], []
        for regime in regimes:
            for _ in range(n_per_class):
                X.append(self.sample(regime=regime, mc=True))
                y.append(regime)
        return np.asarray(X), np.asarray(y)



In [4]:

if __name__ == "__main__":
    sim = BearingSignalSimulator()
    X, y = sim.make_dataset(n_per_class=10)
    print("Dataset shape:", X.shape)
    unique, counts = np.unique(y, return_counts=True)
    print(dict(zip(unique, counts)))

Dataset shape: (40, 12000)
{np.str_('ball_fault'): np.int64(10), np.str_('healthy'): np.int64(10), np.str_('inner_race'): np.int64(10), np.str_('outer_race'): np.int64(10)}
